# JSSP Dynamic Programming - Phase 1 (Google Colab)
## Exact Algorithm: Gromicho et al. (2012)

**Instances**: FT06 + LA01-LA20 (21 instances)

**Algorithm Features**:
- State-space DP with dominance pruning (Proposition 2)
- State-space reduction (Proposition 5)
- Antichain-based bounding
- Beam search with BDP pruning

**Configuration**:
- Timeout: 3600s per instance
- Expected: 15-18 instances proven optimal (makespan = BKS)
- Google Drive checkpoint/resume support

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/jssp_dp_phase1'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

In [ ]:
import os
import sys
import time
import logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, Literal
import pandas as pd

# Configuration
OUTPUT_PATH = os.path.join(CHECKPOINT_DIR, "dp_phase1_results.csv")
TIMEOUT = 3600  # seconds per instance

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print(f"Output: {OUTPUT_PATH}")
print(f"Timeout: {TIMEOUT}s per instance")
print("Setup complete")

In [ ]:
# State Space Implementation
"""
state_space.py — State Representation & Reduction for the JSSP DP Solver

Implements the following concepts from Gromicho et al. (2012):
  - Operation, Job, Machine data structures
  - Ordered partial sequences  (Definition 3)
  - Expansion rules            (Definition 4)
  - State key representation using job progress vectors
    — reduces 2^{nm} possible subsets to (m+1)^n valid subsets
  - Earliest start / completion time computations
  - ξ(T,o) aptitude values     (Section 3.1)
"""

from __future__ import annotations

import logging
from dataclasses import dataclass, field
from typing import Optional

logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# Core data structures
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class Operation:
    """
    A single operation in the JSSP.

    Attributes
    ----------
    job : int          – Job index (0-based)
    op_index : int     – Position within the job's route (0-based)
    machine : int      – Machine on which this operation runs
    processing_time : int – Duration
    global_id : int    – Unique ID across all operations (row-major)
    """

    job: int
    op_index: int
    machine: int
    processing_time: int
    global_id: int

    def __repr__(self) -> str:
        return (
            f"Op(j{self.job},op{self.op_index},m{self.machine},"
            f"p={self.processing_time},id={self.global_id})"
        )


@dataclass
class JSSPInstance:
    """
    Holds a complete JSSP instance.

    Notation (following Gromicho 2012):
      - J = {j_1, ..., j_n}  jobs
      - M = {m_1, ..., m_m}  machines
      - p_{ij}: processing time of job j on machine i
      - π_j(i): the i-th machine that job j visits
      - O = {o_1, ..., o_{nm}}: all operations

    Parameters
    ----------
    name : str
    n_jobs : int
    n_machines : int
    jobs : list[list[tuple[int,int]]]
        jobs[j] = [(machine, processing_time), ...] in visit order
    """

    name: str
    n_jobs: int
    n_machines: int
    jobs: list[list[tuple[int, int]]]

    # Built during __post_init__
    operations: list[Operation] = field(default_factory=list, init=False)
    ops_by_job: dict[int, list[Operation]] = field(
        default_factory=dict, init=False
    )
    pmax: int = field(default=0, init=False)

    def __post_init__(self) -> None:
        self.operations = []
        self.ops_by_job = {}
        self.pmax = 0

        gid = 0
        for j in range(self.n_jobs):
            self.ops_by_job[j] = []
            for idx, (machine, ptime) in enumerate(self.jobs[j]):
                op = Operation(
                    job=j,
                    op_index=idx,
                    machine=machine,
                    processing_time=ptime,
                    global_id=gid,
                )
                self.operations.append(op)
                self.ops_by_job[j].append(op)
                self.pmax = max(self.pmax, ptime)
                gid += 1
        
        # Optimization #2: Precompute remaining load for _lb_estimate
        # remaining_load[j][k][m] = total processing time of job j on machine m
        # from operation k onwards
        self.remaining_load: list[list[list[int]]] = []
        for j in range(self.n_jobs):
            ops = self.ops_by_job[j]
            n = len(ops)
            # Suffix sum by machine
            table = [[0] * self.n_machines for _ in range(n + 1)]
            for k in range(n - 1, -1, -1):
                op = ops[k]
                for m in range(self.n_machines):
                    table[k][m] = table[k + 1][m]
                table[k][op.machine] += op.processing_time
            self.remaining_load.append(table)

    def get_operation(self, job: int, op_index: int) -> Operation:
        """Return the operation for job *job* at position *op_index*."""
        return self.ops_by_job[job][op_index]

    def get_expandable_operations(
        self, current_ops: frozenset[Operation]
    ) -> list[Operation]:
        """
        ε(S): Return all operations that can expand S.

        An operation o can expand S iff:
          - o ∉ S
          - All predecessors of o in the same job are already in S
        """
        # Determine progress per job  (how many ops already scheduled)
        progress = self._job_progress(current_ops)
        result: list[Operation] = []
        for j in range(self.n_jobs):
            next_idx = progress[j]
            if next_idx < self.n_machines:
                result.append(self.ops_by_job[j][next_idx])
        return result

    def full_state_key(self) -> "StateKey":
        """Return the StateKey corresponding to all operations scheduled."""
        return StateKey(
            progress=tuple([self.n_machines] * self.n_jobs)
        )

    def ops_from_state_key(self, key: "StateKey") -> frozenset[Operation]:
        """
        Reconstruct the scheduled operation set from a progress-vector key.

        Since valid states always contain the first k_j operations of each job,
        the full set can be recovered directly from the progress vector.
        """
        ops: list[Operation] = []
        for job, scheduled_count in enumerate(key.progress):
            ops.extend(self.ops_by_job[job][:scheduled_count])
        return frozenset(ops)

    def _job_progress(
        self, current_ops: frozenset[Operation]
    ) -> list[int]:
        """For each job, count how many operations are in current_ops."""
        progress = [0] * self.n_jobs
        for op in current_ops:
            progress[op.job] += 1
        return progress


# ---------------------------------------------------------------------------
# State Key  —  compact representation of operation subset S
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class StateKey:
    """
    Compact representation of a subset S ⊆ O.

    Because each job's operations must be added in order, S is fully
    characterised by a *progress vector*  (k_1, ..., k_n)  where k_j
    is the number of operations of job j already in S.

    This reduces 2^{nm} possible subsets to at most (m+1)^n valid ones
    (Section 4.1, Gromicho 2012).
    """

    progress: tuple[int, ...]  # (k_1, k_2, ..., k_n)

    @property
    def size(self) -> int:
        """Total number of operations in this state."""
        return sum(self.progress)

    @classmethod
    def from_ops(
        cls, ops: frozenset[Operation], n_jobs: int | None = None
    ) -> "StateKey":
        """Build a StateKey from a set of operations."""
        if n_jobs is None:
            if not ops:
                return cls(progress=())
            n_jobs = max(op.job for op in ops) + 1

        progress = [0] * n_jobs
        for op in ops:
            progress[op.job] += 1
        return cls(progress=tuple(progress))

    @property
    def ops(self) -> frozenset[Operation]:
        """
        NOTE: This is a placeholder — in practice we should not
        reconstruct the frozenset from the key.  The solver passes
        op sets alongside keys where needed.
        """
        raise NotImplementedError(
            "Use the explicit op set tracked by the solver."
        )


# We override the StateKey.ops property so that callers that need the
# actual frozenset pass it through the solver.  For the state key
# used purely as a dict key, the progress tuple suffices.


# ---------------------------------------------------------------------------
# Ordered Partial Sequence
# ---------------------------------------------------------------------------
@dataclass
class OrderedPartialSequence:
    """
    An ordered partial sequence T as defined in Definition 3 of
    Gromicho (2012).

    Stores:
      - The list of operations in order
      - Cmax(T): makespan of the partial schedule
      - Machine completion times: when each machine finishes its last op in T
      - Job completion times: when each job finishes its last op in T
      - The last machine i*(T) for ordering tie-breaks
      - job_progress: how many ops of each job are scheduled (avoids recomputation)
    """

    operations: list[Operation]
    cmax: int
    machine_end: dict[int, int]    # machine → completion time
    job_end: dict[int, int]        # job → completion time
    last_machine: int              # i*(T) — machine of last operation
    job_progress: list[int]        # job_progress[j] = #ops of job j in sequence

    @classmethod
    def create_single(
        cls, op: Operation, inst: JSSPInstance
    ) -> "OrderedPartialSequence":
        """Create a sequence containing a single operation."""
        machine_end = {m: 0 for m in range(inst.n_machines)}
        job_end = {j: 0 for j in range(inst.n_jobs)}
        machine_end[op.machine] = op.processing_time
        job_end[op.job] = op.processing_time
        progress = [0] * inst.n_jobs
        progress[op.job] = 1
        return cls(
            operations=[op],
            cmax=op.processing_time,
            machine_end=machine_end,
            job_end=job_end,
            last_machine=op.machine,
            job_progress=progress,
        )

    def earliest_start(self, op: Operation, inst: JSSPInstance) -> int:
        """
        c(T, o): earliest time operation *op* can start if appended to T.

        Must wait for:
          1) The machine to be free  → machine_end[op.machine]
          2) The job's previous op   → job_end[op.job]
        """
        return max(
            self.machine_end.get(op.machine, 0),
            self.job_end.get(op.job, 0),
        )

    def xi_value(self, op: Operation, inst: JSSPInstance) -> int:
        """
        ξ(T, o) — the aptitude value (Section 3.1):

          ξ(T,o) = c(T,o) + p(o)   if o ∈ Z(T)    [ordered expansion]
                 = Cmax(T) + p(o)   otherwise

        Here we compute it generally; the caller decides if o ∈ Z(T).
        """
        c = self.earliest_start(op, inst)
        completion = c + op.processing_time

        # Check if T+o would be ordered
        if completion > self.cmax:
            return completion
        elif completion == self.cmax and op.machine > self.last_machine:
            return completion
        else:
            # o ∉ Z(T): use the upper bound
            return self.cmax + op.processing_time

    def expand_with(
        self, op: Operation, inst: JSSPInstance
    ) -> "OrderedPartialSequence":
        """
        Create a new ordered partial sequence T + o.

        Pre-condition: o ∈ Z(T)  (the caller must ensure this).
        """
        new_ops = self.operations + [op]

        # Compute new machine/job end times
        new_machine_end = dict(self.machine_end)
        new_job_end = dict(self.job_end)

        start = max(
            new_machine_end.get(op.machine, 0),
            new_job_end.get(op.job, 0),
        )
        end = start + op.processing_time

        new_machine_end[op.machine] = end
        new_job_end[op.job] = end

        new_cmax = max(self.cmax, end)

        new_progress = list(self.job_progress)
        new_progress[op.job] += 1

        return OrderedPartialSequence(
            operations=new_ops,
            cmax=new_cmax,
            machine_end=new_machine_end,
            job_end=new_job_end,
            last_machine=op.machine,
            job_progress=new_progress,
        )

    def get_op_set(self) -> frozenset[Operation]:
        """Return the frozenset of all operations in this sequence."""
        return frozenset(self.operations)

    def __repr__(self) -> str:
        ids = [op.global_id for op in self.operations]
        return f"Seq(ops={ids}, Cmax={self.cmax})"

print('State space module loaded')

In [ ]:
# Dominance Rules
"""
dominance.py — Dominance Rules & Antichain-Based Pruning

Implements the dominance framework from Gromicho et al. (2012):

  Proposition 2 (Dominance):
    If T₁, T₂ ∈ X(S) and ξ(T₂, o) ≤ ξ(T₁, o)  ∀o ∈ ε(S)
    with at least one strict inequality, then T₂ ≻ T₁.

  Corollary 2:
    Every ordered completion of T₁ is dominated by the same
    (possibly unordered) completion of T₂.

  Proposition 5 (State-Space Reduction):
    If ∃ machine i such that:
      (i)  ∃ oₙ ∈ ε(S,i) with oₙ ∉ Z(T), AND
      (ii) ∀ o ∈ ε(S,i): ξ(T,o) = Cmax(T) + p(o)
    then T can be pruned (an optimal sequence not starting with T exists).

  Antichain bounding (Propositions 7, 8):
    |X̂(S)| = O(pmax^n / √n)  —  from antichain analysis on multisets.
"""

from __future__ import annotations

import logging
logger = logging.getLogger(__name__)


class DominanceChecker:
    """
    Implements dominance comparison and state-space reduction.
    """

    def __init__(self, instance: "JSSPInstance") -> None:
        self.instance = instance

    # ------------------------------------------------------------------
    # Proposition 2: Dominance comparison
    # ------------------------------------------------------------------
    def compare(
        self,
        t_a: "OrderedPartialSequence",
        t_b: "OrderedPartialSequence",
        expandable_ops: list["Operation"],
        inst: "JSSPInstance",
    ) -> Literal["dominates", "dominated", "equal", "incomparable"]:
        """
        Compare two ordered partial sequences over the same operation set S.

        Returns
        -------
        "dominates"    : t_a ≻ t_b  (t_a is strictly better)
        "dominated"    : t_b ≻ t_a  (t_b is strictly better)
        "equal"        : t_a ≅ t_b  (identical ξ-vectors)
        "incomparable" : neither dominates the other (antichain members)
        """
        if not expandable_ops:
            # No expandable ops → compare by Cmax directly
            if t_a.cmax < t_b.cmax:
                return "dominates"
            elif t_a.cmax > t_b.cmax:
                return "dominated"
            else:
                return "equal"

        a_leq_b = True   # ξ(t_a, o) ≤ ξ(t_b, o)  ∀o
        b_leq_a = True   # ξ(t_b, o) ≤ ξ(t_a, o)  ∀o
        a_strict = False  # at least one strict inequality for a
        b_strict = False  # at least one strict inequality for b

        for op in expandable_ops:
            xi_a = t_a.xi_value(op, inst)
            xi_b = t_b.xi_value(op, inst)

            if xi_a > xi_b:
                a_leq_b = False
                b_strict = True
            elif xi_a < xi_b:
                b_leq_a = False
                a_strict = True

        if a_leq_b and b_leq_a:
            # All ξ values equal → T₁ ≅ T₂
            return "equal"
        elif a_leq_b and a_strict:
            # ξ(t_a, o) ≤ ξ(t_b, o) ∀o with ≥1 strict  → t_a ≻ t_b
            return "dominates"
        elif b_leq_a and b_strict:
            # ξ(t_b, o) ≤ ξ(t_a, o) ∀o with ≥1 strict  → t_b ≻ t_a
            return "dominated"
        else:
            return "incomparable"

    # ------------------------------------------------------------------
    # Proposition 5: State-space reduction
    # ------------------------------------------------------------------
    def check_state_reduction(
        self,
        seq: "OrderedPartialSequence",
        op_set: frozenset["Operation"],
        inst: "JSSPInstance",
    ) -> bool:
        """
        Check if the sequence *seq* can be pruned via Proposition 5.

        Condition: ∃ machine i such that
          (i)  ∃ oₙ ∈ ε(S, i) : oₙ ∉ Z(T)
          (ii) ∀ o ∈ ε(S, i) : ξ(T, o) = Cmax(T) + p(o)

        If true, there is an optimal solution that does NOT start with T,
        so T can safely be skipped for expansion.

        Returns
        -------
        True if the sequence should be pruned.
        """
        expandable = inst.get_expandable_operations(op_set)
        if not expandable:
            return False

        # Group expandable operations by machine: ε(S, i)
        by_machine: dict[int, list["Operation"]] = {}
        for op in expandable:
            by_machine.setdefault(op.machine, []).append(op)

        for machine_id, ops_on_machine in by_machine.items():
            # Condition (i): ∃ oₙ ∈ ε(S, i) such that oₙ ∉ Z(T)
            has_non_ordered = False
            # Condition (ii): ∀ o ∈ ε(S, i): ξ(T, o) = Cmax(T) + p(o)
            all_maximal = True

            for op in ops_on_machine:
                xi_val = seq.xi_value(op, inst)
                expected_max = seq.cmax + op.processing_time

                if xi_val != expected_max:
                    all_maximal = False
                    break

                # Check if o ∉ Z(T)
                c = seq.earliest_start(op, inst)
                completion = c + op.processing_time
                is_ordered = (
                    completion > seq.cmax
                    or (completion == seq.cmax and op.machine > seq.last_machine)
                )
                if not is_ordered:
                    has_non_ordered = True

            if all_maximal and has_non_ordered:
                return True

        return False

print('Dominance module loaded')

In [ ]:
# Dynamic Programming Solver
"""
dp_solver.py — Core Dynamic Programming Algorithm for the Job-Shop Scheduling Problem

Implements Algorithm 1 from:
  Gromicho, van Hoorn, Saldanha-da-Gama, Timmer (2012)
  "Solving the job-shop scheduling problem optimally by dynamic programming"
  Computers & Operations Research 39, 2968–2977.

Based on the Bellman equation adapted from:
  Held & Karp (1962)
  "A dynamic programming approach to sequencing problems"
  SIAM Journal of Applied Mathematics 10, 196–210.

Complexity: O((pmax)^{2n} (m+1)^n)  —  exponentially better than brute force O((n!)^m)
"""

from __future__ import annotations

import json
import logging
import signal
import time
import tracemalloc
from dataclasses import dataclass, field
from typing import Optional

# All classes already defined in previous cells
# DominanceChecker already defined in previous cell

logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# Timeout mechanism
# ---------------------------------------------------------------------------
class TimeoutError(Exception):
    """Raised when computation exceeds the allowed time budget."""
    pass


def _timeout_handler(signum, frame):
    import signal
    signal.alarm(0)  # Tắt alarm NGAY LẬP TỨC trước khi raise
    raise TimeoutError("Computation exceeded timeout limit.")


# ---------------------------------------------------------------------------
# Result container
# ---------------------------------------------------------------------------
@dataclass
class DPResult:
    """Stores the result produced by the DP solver."""

    instance_name: str = ""
    size: str = ""
    optimal_makespan: int | None = None
    best_makespan: int | None = None
    gap_percent: float | None = None
    computation_time_seconds: float = 0.0
    states_explored: int = 0
    memory_mb: float = 0.0
    optimal_proven: bool = False
    schedule: list[dict] | None = None
    sequence: list[int] | None = None
    timed_out: bool = False

    def to_json(self) -> str:
        return json.dumps(self.__dict__, indent=2, default=str)


@dataclass
class IncumbentSolution:
    """Best complete schedule found so far."""

    makespan: int
    schedule: list[dict]
    sequence: list[int]
    source: str = ""


# ---------------------------------------------------------------------------
# Main DP Solver  (Algorithm 1 — Gromicho et al. 2012)
# ---------------------------------------------------------------------------
class DPSolver:
    """
    Exact DP solver for the Job-Shop Scheduling Problem.

    Implements the Bellman equation for JSSP with:
      - Ordered partial sequences (Definition 3, 4)
      - Dominance pruning        (Proposition 2)
      - State-space reduction     (Proposition 5)
      - Antichain bounding        (Propositions 7, 8)
    """

    def __init__(
        self,
        instance: JSSPInstance,
        timeout: int = 3600,
        enable_state_reduction: bool = True,
        log_interval: int = 100_000,
        max_width: int = 0,
    ) -> None:
        self.instance = instance
        self.timeout = timeout
        self.enable_state_reduction = enable_state_reduction
        self.log_interval = log_interval
        # max_width > 0 enables Bounded DP (BDP): keep only the best
        # max_width states per level, pruning by lower-bound estimate.
        # 0 = unlimited (pure DP, exact but memory-intensive).
        self.max_width = max_width

        self.dominance = DominanceChecker(instance)
        self.job_symmetry_map: list[int] = self._find_identical_jobs()
        
        # Signal handling flag
        self._use_signal = False

        # X̂(S)  →  dict  mapping  StateKey  →  list[OrderedPartialSequence]
        self.xi_hat: dict[StateKey, list[OrderedPartialSequence]] = {}
        
        # Optimization #3: Index xi_hat by level for O(1) lookup
        from collections import defaultdict
        self.xi_hat_by_level: dict[int, set[StateKey]] = defaultdict(set)

        self.states_explored: int = 0
        self.total_sequences_generated: int = 0
        self.total_dominated_pruned: int = 0
        self.total_state_reduced: int = 0
        self.total_bdp_pruned: int = 0
        self._start_time: float = 0.0
        self.total_bound_pruned: int = 0
        self.search_completed: bool = False
        self.best_complete: IncumbentSolution | None = None

    # ------------------------------------------------------------------
    # Public entry point
    # ------------------------------------------------------------------
    def solve(self, known_optimal: int | None = None) -> DPResult:
        """
        Run Algorithm 1 from Gromicho (2012).

        Parameters
        ----------
        known_optimal : int | None
            If provided, used to compute the gap.

        Returns
        -------
        DPResult
        """
        result = DPResult(
            instance_name=self.instance.name,
            size=f"{self.instance.n_jobs}x{self.instance.n_machines}",
            optimal_makespan=known_optimal,
        )

        tracemalloc.start()
        self._start_time = time.perf_counter()
        self._initialise_incumbent()

        # Set up timeout via SIGALRM (Unix) or fallback polling
        self._use_signal = _try_set_alarm(self.timeout)

        try:
            self._run_algorithm()
        except TimeoutError:
            # Tắt alarm NGAY LẬP TỨC trước khi làm bất cứ gì
            if self._use_signal:
                signal.alarm(0)
            logger.warning("Timeout reached after %d seconds.", self.timeout)
            result.timed_out = True
        except Exception:
            # Tắt alarm nếu có lỗi khác
            if self._use_signal:
                signal.alarm(0)
            logger.exception("Solver encountered an error.")
            raise
        finally:
            # Đảm bảo alarm được tắt (double-check)
            if self._use_signal:
                signal.alarm(0)

        elapsed = time.perf_counter() - self._start_time
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()

        result.computation_time_seconds = round(elapsed, 3)
        result.memory_mb = round(peak / (1024 * 1024), 1)
        result.states_explored = self.states_explored

        # Extract the optimal solution from X̂(O)
        full_key = self.instance.full_state_key()
        if full_key in self.xi_hat and self.xi_hat[full_key]:
            best = self.xi_hat[full_key][0]
            result.best_makespan = best.cmax
            result.sequence = [op.global_id for op in best.operations]
            result.schedule = self._build_schedule(best)
            # Optimal only if: no timeout AND (no BDP pruning OR found == known optimal)
            bdp_active = self.max_width > 0 and self.total_bdp_pruned > 0
            exact_match = (known_optimal is not None and best.cmax == known_optimal)
            result.optimal_proven = (not result.timed_out) and (not bdp_active or exact_match)

            if known_optimal is not None and result.best_makespan is not None:
                result.gap_percent = round(
                    100.0
                    * (result.best_makespan - known_optimal)
                    / known_optimal,
                    2,
                )
        else:
            # Try to find any complete sequence in the DP state store
            best_seq = self._find_best_complete_sequence()

            # Fallback: use the greedy incumbent from _initialise_incumbent
            # (always available — set before the main loop starts)
            if best_seq is None and self.best_complete is not None:
                result.best_makespan = self.best_complete.makespan
                result.schedule = self.best_complete.schedule
                result.sequence = self.best_complete.sequence
            elif best_seq is not None:
                result.best_makespan = best_seq.cmax
                result.sequence = [op.global_id for op in best_seq.operations]
                result.schedule = self._build_schedule(best_seq)

            if result.best_makespan is not None and known_optimal is not None:
                result.gap_percent = round(
                    100.0 * (result.best_makespan - known_optimal) / known_optimal, 2
                )

        return result

    # ------------------------------------------------------------------
    # Algorithm 1 — main loop
    # ------------------------------------------------------------------
    def _run_algorithm(self) -> None:
        """
        Algorithm 1 from Gromicho et al. (2012):

        1) Initialize: for each first operation o of each job,
           create a single-operation sequence and store in X̂({o}).
        2) For l = 1 to n*m:
             For each S with |S| = l:
               For each T₁ ∈ X̂(S):
                 For each o ∈ Z(T₁):            # ordered expansions
                   T₁' = T₁ + o
                   Insert T₁' into X̂(S ∪ {o}) applying dominance rules
        """
        inst = self.instance
        n_ops = inst.n_jobs * inst.n_machines  # total operations

        # --- Step 1: Initialise with first operations of each job ---
        first_ops = inst.get_expandable_operations(frozenset())
        for op in first_ops:
            seq = OrderedPartialSequence.create_single(op, inst)
            key = StateKey.from_ops(frozenset([op]), n_jobs=inst.n_jobs)
            self.xi_hat.setdefault(key, [])
            self.xi_hat[key].append(seq)
            # Optimization #3: Track level
            self.xi_hat_by_level[key.size].add(key)
            self.states_explored += 1
            self.total_sequences_generated += 1

        logger.info(
            "Initialised %d single-operation sequences.", len(first_ops)
        )

        full_key = inst.full_state_key()

        # --- Step 2: Expand level by level ---
        for length in range(1, n_ops):
            # Optimization #3: O(1) lookup by level instead of O(total_states) filter
            keys_at_length = list(self.xi_hat_by_level.get(length, set()))

            if not keys_at_length:
                continue

            logger.debug(
                "Level %d: %d state keys to expand.", length, len(keys_at_length)
            )

            for state_key in keys_at_length:
                sequences = list(self.xi_hat.get(state_key, []))
                op_set = inst.ops_from_state_key(state_key)

                for seq in sequences:
                    # Check timeout via polling (fallback for Windows)
                    if (time.perf_counter() - self._start_time) > self.timeout:
                        # Tắt alarm nếu đang dùng signal
                        if self._use_signal:
                            signal.alarm(0)
                        raise TimeoutError(
                            "Computation exceeded timeout limit."
                        )

                    # --- State-space reduction (Proposition 5) ---
                    if self.enable_state_reduction:
                        if self.dominance.check_state_reduction(seq, op_set, inst):
                            self.total_state_reduced += 1
                            continue

                    # Compute Z(T) — operations giving ordered expansions
                    expandable = inst.get_expandable_operations(op_set)
                    ordered_expansions = self._get_ordered_expansions(
                        seq, expandable
                    )

                    for op in ordered_expansions:
                        new_seq = seq.expand_with(op, inst)
                        self.total_sequences_generated += 1
                        
                        # Optimization #1: Compute new_key directly from progress vector
                        # O(1) instead of O(nm) from StateKey.from_ops
                        new_progress = list(state_key.progress)
                        new_progress[op.job] += 1
                        new_key = StateKey(progress=tuple(new_progress))

                        self._insert_with_dominance(new_key, new_seq)

                        # Update incumbent AFTER insertion so xi_hat[full_key]
                        # is populated before bound pruning tightens the cutoff.
                        if new_key == full_key:
                            if (
                                self.best_complete is None
                                or new_seq.cmax < self.best_complete.makespan
                            ):
                                self.best_complete = IncumbentSolution(
                                    makespan=new_seq.cmax,
                                    schedule=self._build_schedule(new_seq),
                                    sequence=[
                                        op2.global_id for op2 in new_seq.operations
                                    ],
                                    source="dp",
                                )
                        self.states_explored += 1

                        if self.states_explored % self.log_interval == 0:
                            elapsed = time.perf_counter() - self._start_time
                            logger.info(
                                "States explored: %d | Sequences in store: %d | "
                                "Dominated pruned: %d | State reduced: %d | "
                                "Bound pruned: %d | Elapsed: %.1fs",
                                self.states_explored,
                                sum(len(v) for v in self.xi_hat.values()),
                                self.total_dominated_pruned,
                                self.total_state_reduced,
                                self.total_bound_pruned,
                                elapsed,
                            )

            # Free states at this level — all expansions already generated
            for k in keys_at_length:
                if k in self.xi_hat:
                    del self.xi_hat[k]
            # Optimization #3: Also remove level index
            self.xi_hat_by_level.pop(length, None)

            # BDP: prune newly generated states at level (length+1)
            if self.max_width > 0 and length + 1 < n_ops:
                self._bdp_prune(length + 1)

        logger.info(
            "Algorithm complete. Total states explored: %d, "
            "Total sequences generated: %d, Dominated pruned: %d, "
            "State reduced: %d, Bound pruned: %d, BDP pruned: %d",
            self.states_explored,
            self.total_sequences_generated,
            self.total_dominated_pruned,
            self.total_state_reduced,
            self.total_bound_pruned,
            self.total_bdp_pruned,
        )

    # ------------------------------------------------------------------
    # Job symmetry detection (called once at init)
    # ------------------------------------------------------------------
    def _find_identical_jobs(self) -> list[int]:
        """
        For each job j, return the index of its canonical representative
        (lowest-index job with the same machine/processing-time sequence).
        Used for symmetry-breaking pruning.
        """
        inst = self.instance
        groups: dict[tuple, int] = {}
        mapping: list[int] = []
        for j in range(inst.n_jobs):
            sig = tuple(
                (op.machine, op.processing_time) for op in inst.ops_by_job[j]
            )
            if sig not in groups:
                groups[sig] = j
            mapping.append(groups[sig])
        return mapping

    # ------------------------------------------------------------------
    # Ordered expansion check  (Definition 3 / Z(T))
    # ------------------------------------------------------------------
    def _get_ordered_expansions(
        self,
        seq: OrderedPartialSequence,
        expandable_ops: list[Operation],
    ) -> list[Operation]:
        """
        Return the subset Z(T) ⊆ ε(S):
        An operation o ∈ ε(S) is in Z(T) iff T+o is an ordered partial sequence.

        From the paper:
          o ∈ Z(T)  iff  c(T,o) + p(o) > Cmax(T)
                         OR  (c(T,o)+p(o) == Cmax(T)  AND  m(o) > i*(T))

        where i*(T) is the machine of the last operation in T.
        """
        # Optimization #4: Symmetry breaking in O(n) instead of O(n²)
        seen_canonical = set()
        result: list[Operation] = []
        
        for op in expandable_ops:
            j = op.job
            # Symmetry breaking: if job j hasn't started and we've already
            # seen its canonical representative, skip it
            if seq.job_progress[j] == 0:
                canon = self.job_symmetry_map[j]
                if canon in seen_canonical:
                    continue
                seen_canonical.add(canon)

            c_to = seq.earliest_start(op, self.instance)
            completion = c_to + op.processing_time
            if completion > seq.cmax:
                result.append(op)
            elif completion == seq.cmax and op.machine > seq.last_machine:
                result.append(op)
        return result

    # ------------------------------------------------------------------
    # Dominance-aware insertion  (Proposition 2)
    # ------------------------------------------------------------------
    def _insert_with_dominance(
        self, key: StateKey, new_seq: OrderedPartialSequence
    ) -> None:
        """
        Insert *new_seq* into X̂(S) identified by *key*, applying
        the dominance relation from Proposition 2:

          T₂ ≻ T₁   iff   ξ(T₂,o) ≤ ξ(T₁,o) ∀o ∈ ε(S), with ≥1 strict.

        If new_seq is dominated by any existing sequence, discard it.
        Otherwise, add it and remove any existing sequences it dominates.
        """
        # Bound pruning: cmax of a partial sequence is itself a lower bound —
        # the final makespan cannot be less than cmax.  If it already meets or
        # exceeds the best known complete solution, this branch cannot improve.
        if self.best_complete is not None and new_seq.cmax >= self.best_complete.makespan:
            self.total_bound_pruned += 1
            return

        existing = self.xi_hat.get(key, [])
        expandable = self.instance.get_expandable_operations(
            self.instance.ops_from_state_key(key)
        )

        # Check if new_seq is dominated by an existing sequence
        for ex_seq in existing:
            rel = self.dominance.compare(ex_seq, new_seq, expandable, self.instance)
            if rel == "dominates" or rel == "equal":
                # ex_seq dominates or equals new_seq → discard new_seq
                self.total_dominated_pruned += 1
                return

        # new_seq survives; now remove any existing sequences it dominates
        surviving: list[OrderedPartialSequence] = []
        for ex_seq in existing:
            rel = self.dominance.compare(new_seq, ex_seq, expandable, self.instance)
            if rel == "dominates":
                self.total_dominated_pruned += 1
            else:
                surviving.append(ex_seq)

        surviving.append(new_seq)
        self.xi_hat[key] = surviving
        
        # Optimization #3: Track key by level
        if key not in self.xi_hat_by_level[key.size]:
            self.xi_hat_by_level[key.size].add(key)

    # ------------------------------------------------------------------
    # BDP helpers
    # ------------------------------------------------------------------
    def _lb_estimate(self, seq: OrderedPartialSequence) -> int:
        """
        Lower bound on final makespan from partial state seq.

        For each machine m:  lb_m = machine_end[m] + remaining work on m
        For each job j:      lb_j = job_end[j]     + remaining work of j
        Returns max of all lb values and current cmax.
        
        Optimization #2: Uses precomputed remaining_load table for O(n·m) instead of O(n·m²).
        """
        inst = self.instance
        progress = seq.job_progress

        lb = seq.cmax
        
        # Machine bound: use precomputed remaining_load
        for m in range(inst.n_machines):
            remaining = sum(
                inst.remaining_load[j][progress[j]][m]
                for j in range(inst.n_jobs)
            )
            lb = max(lb, seq.machine_end.get(m, 0) + remaining)

        # Job bound: sum remaining operations for each job
        for j in range(inst.n_jobs):
            remaining = sum(
                inst.ops_by_job[j][k].processing_time
                for k in range(progress[j], inst.n_machines)
            )
            lb = max(lb, seq.job_end.get(j, 0) + remaining)

        return lb

    def _bdp_prune(self, level: int) -> None:
        """Keep only the best max_width states at *level* by LB estimate."""
        keys_at_level = [k for k in list(self.xi_hat.keys()) if k.size == level]
        all_seqs = [
            (self._lb_estimate(s), kid, s)
            for kid in keys_at_level
            for s in self.xi_hat[kid]
        ]
        if len(all_seqs) <= self.max_width:
            return

        all_seqs.sort(key=lambda x: x[0])
        pruned = len(all_seqs) - self.max_width
        self.total_bdp_pruned += pruned

        new_level: dict[StateKey, list[OrderedPartialSequence]] = {}
        for _, kid, s in all_seqs[:self.max_width]:
            new_level.setdefault(kid, []).append(s)

        for k in keys_at_level:
            if k in new_level:
                self.xi_hat[k] = new_level[k]
            else:
                del self.xi_hat[k]

        logger.debug("BDP level %d: kept %d / %d states", level, self.max_width, len(all_seqs))

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------
    def _build_schedule(
        self, seq: OrderedPartialSequence
    ) -> list[dict]:
        """Convert a complete ordered sequence into a schedule."""
        inst = self.instance
        machine_time: dict[int, int] = {m: 0 for m in range(inst.n_machines)}
        job_time: dict[int, int] = {j: 0 for j in range(inst.n_jobs)}

        schedule: list[dict] = []
        for op in seq.operations:
            start = max(machine_time[op.machine], job_time[op.job])
            end = start + op.processing_time
            machine_time[op.machine] = end
            job_time[op.job] = end
            schedule.append(
                {
                    "job": op.job,
                    "machine": op.machine,
                    "operation": op.op_index,
                    "start": start,
                    "end": end,
                    "processing_time": op.processing_time,
                }
            )
        return schedule

    def _initialise_incumbent(self) -> None:
        """
        Build an initial incumbent solution using a greedy dispatching rule
        (Shortest Processing Time first) to seed the best-so-far makespan.
        This is used purely to enable tighter bound pruning during the DP.
        """
        inst = self.instance
        machine_time: dict[int, int] = {m: 0 for m in range(inst.n_machines)}
        job_time: dict[int, int] = {j: 0 for j in range(inst.n_jobs)}
        job_progress = [0] * inst.n_jobs

        schedule: list[dict] = []
        total_ops = inst.n_jobs * inst.n_machines

        for _ in range(total_ops):
            # Collect all currently schedulable operations (next op for each job)
            candidates = [
                inst.ops_by_job[j][job_progress[j]]
                for j in range(inst.n_jobs)
                if job_progress[j] < inst.n_machines
            ]
            # Sort by (earliest possible start + processing time) — SPT-based greedy
            candidates.sort(
                key=lambda op: max(machine_time[op.machine], job_time[op.job])
                + op.processing_time
            )
            op = candidates[0]
            start = max(machine_time[op.machine], job_time[op.job])
            end = start + op.processing_time
            machine_time[op.machine] = end
            job_time[op.job] = end
            job_progress[op.job] += 1
            schedule.append({
                "job": op.job, "machine": op.machine, "operation": op.op_index,
                "start": start, "end": end, "processing_time": op.processing_time,
            })

        makespan = max(machine_time.values())
        self.best_complete = IncumbentSolution(
            makespan=makespan,
            schedule=schedule,
            sequence=[s["job"] * inst.n_machines + s["operation"] for s in schedule],
            source="greedy_spt",
        )
        logger.info("Greedy incumbent makespan: %d", makespan)

    def _find_best_complete_sequence(self) -> OrderedPartialSequence | None:
        """Find the best complete sequence across all states (fallback)."""
        total_ops = self.instance.n_jobs * self.instance.n_machines
        best: OrderedPartialSequence | None = None
        for key, seqs in self.xi_hat.items():
            if key.size == total_ops:
                for s in seqs:
                    if best is None or s.cmax < best.cmax:
                        best = s
        return best


# ---------------------------------------------------------------------------
# Signal-based alarm (Unix only)
# ---------------------------------------------------------------------------
def _try_set_alarm(timeout: int) -> bool:
    """Attempt to set SIGALRM; return True if successful."""
    try:
        signal.signal(signal.SIGALRM, _timeout_handler)
        signal.alarm(timeout)
        return True
    except (AttributeError, ValueError):
        # Windows or non-main thread
        return False

print('DP solver module loaded')

In [ ]:
# Best Known Solutions
BKS = {
    "FT06": 55,
    "LA01": 666,
    "LA02": 655,
    "LA03": 597,
    "LA04": 590,
    "LA05": 593,
    "LA06": 926,
    "LA07": 890,
    "LA08": 863,
    "LA09": 951,
    "LA10": 958,
    "LA11": 1222,
    "LA12": 1039,
    "LA13": 1150,
    "LA14": 1292,
    "LA15": 1207,
    "LA16": 945,
    "LA17": 784,
    "LA18": 848,
    "LA19": 842,
    "LA20": 902
}

# Instance Data
INSTANCES = {
    "FT06": "6 6\n2 1 0 3 1 6 3 7 5 3 4 6\n1 8 2 5 4 10 5 10 0 10 3 4\n2 5 3 4 5 8 0 9 1 1 4 7\n1 5 0 5 2 5 3 3 4 8 5 9\n2 9 1 3 4 5 5 4 0 3 3 1\n1 3 3 3 5 9 0 10 4 4 2 1",
    "LA01": "10 5\n1 21 0 53 4 95 3 55 2 34\n0 21 3 52 4 16 2 26 1 71\n3 39 4 98 1 42 2 31 0 12\n1 77 0 55 4 79 2 66 3 77\n0 83 3 34 2 64 1 19 4 37\n1 54 2 43 4 79 0 92 3 62\n3 69 4 77 1 87 2 87 0 93\n2 38 0 60 1 41 3 24 4 83\n3 17 1 49 4 25 0 44 2 98\n4 77 3 79 2 43 1 75 0 96",
    "LA02": "10 5\n0 20 3 87 1 31 4 76 2 17\n4 25 2 32 0 24 1 18 3 81\n1 72 2 23 4 28 0 58 3 99\n2 86 1 76 4 97 0 45 3 90\n4 27 0 42 3 48 2 17 1 46\n1 67 0 98 4 48 3 27 2 62\n4 28 1 12 3 19 0 80 2 50\n1 63 0 94 2 98 3 50 4 80\n4 14 0 75 2 50 1 41 3 55\n4 72 2 18 1 37 3 79 0 61",
    "LA03": "10 5\n1 23 2 45 0 82 4 84 3 38\n2 21 1 29 0 18 4 41 3 50\n2 38 3 54 4 16 0 52 1 52\n4 37 0 54 2 74 1 62 3 57\n4 57 0 81 1 61 3 68 2 30\n4 81 0 79 1 89 2 89 3 11\n3 33 2 20 0 91 4 20 1 66\n4 24 1 84 0 32 2 55 3  8\n4 56 0  7 3 54 2 64 1 39\n4 40 1 83 0 19 2  8 3  7",
    "LA04": "10 5\n0 12 2 94 3 92 4 91 1  7\n1 19 3 11 4 66 2 21 0 87\n1 14 0 75 3 13 4 16 2 20\n2 95 4 66 0  7 3  7 1 77\n1 45 3  6 4 89 0 15 2 34\n3 77 2 20 0 76 4 88 1 53\n2 74 1 88 0 52 3 27 4  9\n1 88 3 69 0 62 4 98 2 52\n2 61 4  9 0 62 1 52 3 90\n2 54 4  5 3 59 1 15 0 88",
    "LA05": "10 5\n1 72 0 87 4 95 2 66 3 60\n4  5 3 35 0 48 2 39 1 54\n1 46 3 20 2 21 0 97 4 55\n0 59 3 19 4 46 1 34 2 37\n4 23 2 73 3 25 1 24 0 28\n3 28 0 45 4  5 1 78 2 83\n0 53 3 71 1 37 4 29 2 12\n4 12 2 87 3 33 1 55 0 38\n2 49 3 83 1 40 0 48 4  7\n2 65 3 17 0 90 4 27 1 23",
    "LA06": "15 5\n1 21 2 34 4 95 0 53 3 55\n3 52 4 16 1 71 2 26 0 21\n2 31 0 12 1 42 3 39 4 98\n3 77 1 77 4 79 0 55 2 66\n4 37 3 34 2 64 1 19 0 83\n2 43 1 54 0 92 3 62 4 79\n0 93 3 69 1 87 4 77 2 87\n0 60 1 41 2 38 4 83 3 24\n2 98 3 17 4 25 0 44 1 49\n0 96 4 77 3 79 1 75 2 43\n4 28 2 35 0 95 3 76 1 7\n0 61 4 10 2 95 1 9 3 35\n4 59 3 16 1 91 2 59 0 46\n4 43 1 52 0 28 2 27 3 50\n0 87 1 45 2 39 4 9 3 41",
    "LA07": "15 5\n0 47 4 57 1 71 3 96 2 14\n0 75 1 60 4 22 3 79 2 65\n3 32 0 33 2 69 1 31 4 58\n0 44 1 34 4 51 3 58 2 47\n3 29 1 44 0 62 2 17 4 8\n1 15 2 40 0 97 4 38 3 66\n2 58 1 39 0 57 4 20 3 50\n2 57 3 32 4 87 0 63 1 21\n4 56 0 84 2 90 1 85 3 61\n4 15 0 20 1 67 3 30 2 70\n4 84 0 82 1 23 2 45 3 38\n3 50 2 21 0 18 4 41 1 29\n4 16 1 52 0 52 2 38 3 54\n4 37 0 54 3 57 2 74 1 62\n4 57 1 61 0 81 2 30 3 68",
    "LA08": "15 5\n3 92 2 94 0 12 4 91 1 7\n2 21 1 19 0 87 3 11 4 66\n1 14 3 13 0 75 4 16 2 20\n2 95 4 66 0 7 1 77 3 7\n2 34 4 89 3 6 1 45 0 15\n4 88 3 77 2 20 1 53 0 76\n4 9 3 27 0 52 1 88 2 74\n3 69 2 52 0 62 1 88 4 98\n3 90 0 62 4 9 2 61 1 52\n4 5 2 54 3 59 0 88 1 15\n0 41 1 50 4 78 3 53 2 23\n0 38 4 72 2 91 3 68 1 71\n0 45 3 95 4 52 2 25 1 6\n3 30 1 66 0 23 4 36 2 17\n2 95 0 71 3 76 1 8 4 88",
    "LA09": "15 5\n1 66 3 85 2 84 0 62 4 19\n3 59 1 64 2 46 4 13 0 25\n4 88 3 80 1 73 2 53 0 41\n0 14 1 67 2 57 3 74 4 47\n0 84 4 64 2 41 3 84 1 78\n0 63 3 28 1 46 2 26 4 52\n3 10 2 17 4 73 1 11 0 64\n2 67 1 97 3 95 4 38 0 85\n2 95 4 46 0 59 1 65 3 93\n2 43 4 85 3 32 1 85 0 60\n4 49 3 41 2 61 0 66 1 90\n1 17 0 23 3 70 4 99 2 49\n4 40 3 73 0 73 1 98 2 68\n3 57 1 9 2 7 0 13 4 98\n0 37 1 85 2 17 4 79 3 41",
    "LA10": "15 5\n1 58 2 44 3 5 0 9 4 58\n1 89 0 97 4 96 3 77 2 84\n0 77 1 87 2 81 4 39 3 85\n3 57 1 21 2 31 0 15 4 73\n2 48 0 40 1 49 3 70 4 71\n3 34 4 82 2 80 0 10 1 22\n1 91 4 75 0 55 2 17 3 7\n2 62 3 47 1 72 4 35 0 11\n0 64 3 75 4 50 1 90 2 94\n2 67 4 20 3 15 0 12 1 71\n0 52 4 93 3 68 2 29 1 57\n2 70 0 58 1 93 4 7 3 77\n3 27 2 82 1 63 4 6 0 95\n1 87 2 56 4 36 0 26 3 48\n3 76 2 36 0 36 4 15 1 8",
    "LA11": "20 5\n2 34 1 21 0 53 3 55 4 95\n0 21 3 52 1 71 4 16 2 26\n0 12 1 42 2 31 4 98 3 39\n2 66 3 77 4 79 0 55 1 77\n0 83 4 37 3 34 1 19 2 64\n4 79 2 43 0 92 3 62 1 54\n0 93 4 77 2 87 1 87 3 69\n4 83 3 24 1 41 2 38 0 60\n4 25 1 49 0 44 2 98 3 17\n0 96 1 75 2 43 4 77 3 79\n0 95 3 76 1 7 4 28 2 35\n4 10 2 95 0 61 1 9 3 35\n1 91 2 59 4 59 0 46 3 16\n2 27 1 52 4 43 0 28 3 50\n4 9 0 87 3 41 2 39 1 45\n1 54 0 20 4 43 3 14 2 71\n4 33 1 28 3 26 0 78 2 37\n1 89 0 33 2 8 3 66 4 42\n4 84 0 69 2 94 1 74 3 27\n4 81 2 45 1 78 3 69 0 96",
    "LA12": "20 5\n1 23 0 82 4 84 2 45 3 38\n3 50 4 41 1 29 0 18 2 21\n4 16 3 54 1 52 2 38 0 52\n1 62 3 57 4 37 2 74 0 54\n3 68 1 61 2 30 0 81 4 57\n1 89 2 89 3 11 0 79 4 81\n1 66 0 91 3 33 4 20 2 20\n3 8 4 24 2 55 0 32 1 84\n0 7 2 64 1 39 4 56 3 54\n0 19 4 40 3 7 2 8 1 83\n0 63 2 64 3 91 4 40 1 6\n1 42 3 61 4 15 2 98 0 74\n1 80 0 26 3 75 4 6 2 87\n2 39 4 22 0 75 3 24 1 44\n1 15 3 79 4 8 0 12 2 20\n3 26 2 43 0 80 4 22 1 61\n2 62 1 36 0 63 3 96 4 40\n1 33 3 18 0 22 4 5 2 10\n2 64 4 64 0 89 1 96 3 95\n2 18 4 23 3 15 1 38 0 8",
    "LA13": "20 5\n3 60 0 87 1 72 4 95 2 66\n1 54 0 48 2 39 3 35 4 5\n3 20 1 46 0 97 2 21 4 55\n2 37 0 59 3 19 1 34 4 46\n2 73 3 25 1 24 0 28 4 23\n1 78 3 28 2 83 0 45 4 5\n3 71 1 37 2 12 4 29 0 53\n4 12 3 33 1 55 2 87 0 38\n0 48 1 40 2 49 3 83 4 7\n0 90 4 27 2 65 3 17 1 23\n0 62 3 85 1 66 2 84 4 19\n3 59 2 46 4 13 1 64 0 25\n2 53 1 73 3 80 4 88 0 41\n2 57 4 47 0 14 1 67 3 74\n2 41 4 64 3 84 1 78 0 84\n4 52 3 28 2 26 0 63 1 46\n1 11 0 64 3 10 4 73 2 17\n4 38 3 95 0 85 1 97 2 67\n3 93 1 65 2 95 0 59 4 46\n0 60 1 85 2 43 4 85 3 32",
    "LA14": "20 5\n3 5 4 58 2 44 0 9 1 58\n1 89 4 96 0 97 2 84 3 77\n2 81 3 85 1 87 4 39 0 77\n0 15 3 57 4 73 1 21 2 31\n2 48 4 71 3 70 0 40 1 49\n0 10 4 82 3 34 2 80 1 22\n2 17 0 55 1 91 4 75 3 7\n3 47 2 62 1 72 4 35 0 11\n1 90 2 94 4 50 0 64 3 75\n3 15 2 67 0 12 4 20 1 71\n4 93 2 29 0 52 1 57 3 68\n3 77 1 93 0 58 2 70 4 7\n1 63 3 27 0 95 4 6 2 82\n4 36 0 26 3 48 2 56 1 87\n2 36 1 8 4 15 3 76 0 36\n4 78 1 84 3 41 0 30 2 76\n1 78 0 75 4 88 3 13 2 81\n0 54 4 40 2 13 1 82 3 29\n1 26 4 82 0 52 3 6 2 6\n3 54 1 64 0 54 2 32 4 88",
    "LA15": "20 5\n0 6 2 40 1 81 3 37 4 19\n2 40 3 32 0 55 4 81 1 9\n1 46 4 65 2 70 3 55 0 77\n2 21 4 65 0 64 3 25 1 15\n2 85 0 40 1 44 3 24 4 37\n0 89 4 29 1 83 3 31 2 84\n4 59 3 38 1 80 2 30 0 8\n0 80 2 56 1 77 4 41 3 97\n4 56 0 91 3 50 2 71 1 17\n1 40 0 88 4 59 2 7 3 80\n0 45 1 29 2 8 4 77 3 58\n2 36 0 54 3 96 1 9 4 10\n0 28 2 73 1 98 3 92 4 87\n0 70 3 86 2 27 1 99 4 96\n1 95 0 59 4 56 3 85 2 41\n1 81 2 92 4 32 0 52 3 39\n1 7 4 22 2 12 0 88 3 60\n3 45 0 93 2 69 4 49 1 27\n0 21 1 84 2 61 3 68 4 26\n1 82 2 33 4 71 0 99 3 44",
    "LA16": "10 10\n1 21 6 71 9 16 8 52 7 26 2 34 0 53 4 21 3 55 5 95\n4 55 2 31 5 98 9 79 0 12 7 66 1 42 8 77 6 77 3 39\n3 34 2 64 8 62 1 19 4 92 9 79 7 43 6 54 0 83 5 37\n1 87 3 69 2 87 7 38 8 24 9 83 6 41 0 93 5 77 4 60\n2 98 0 44 5 25 6 75 7 43 1 49 4 96 9 77 3 17 8 79\n2 35 3 76 5 28 9 10 4 61 6 9 0 95 8 35 1 7 7 95\n3 16 2 59 0 46 1 91 9 43 8 50 6 52 5 59 4 28 7 27\n1 45 0 87 3 41 4 20 6 54 9 43 8 14 5 9 2 39 7 71\n4 33 2 37 8 66 5 33 3 26 7 8 1 28 6 89 9 42 0 78\n8 69 9 81 2 94 4 96 3 27 0 69 7 45 6 78 1 74 5 84",
    "LA17": "10 10\n4 18 7 21 9 41 2 45 3 38 8 50 5 84 6 29 1 23 0 82\n8 57 5 16 1 52 7 74 2 38 3 54 6 62 9 37 4 54 0 52\n2 30 4 79 3 68 1 61 8 11 6 89 7 89 0 81 9 81 5 57\n0 91 8 8 3 33 7 55 5 20 2 20 4 32 6 84 1 66 9 24\n9 40 0 7 4 19 8 7 6 83 2 64 5 56 3 54 7 8 1 39\n3 91 2 64 5 40 0 63 7 98 4 74 8 61 1 6 6 42 9 15\n1 80 7 39 8 24 3 75 4 75 5 6 6 44 0 26 2 87 9 22\n1 15 7 43 2 20 0 12 8 26 6 61 3 79 9 22 5 8 4 80\n2 62 3 96 4 22 9 5 0 63 6 33 7 10 8 18 1 36 5 40\n1 96 0 89 5 64 3 95 9 23 7 18 8 15 2 64 6 38 4 8",
    "LA18": "10 10\n6 54 0 87 4 48 3 60 7 39 8 35 1 72 5 95 2 66 9 5\n3 20 9 46 6 34 5 55 0 97 8 19 4 59 2 21 7 37 1 46\n4 45 1 24 8 28 0 28 7 83 6 78 5 23 3 25 9 5 2 73\n9 12 1 37 4 38 3 71 8 33 2 12 6 55 0 53 7 87 5 29\n3 83 2 49 6 23 9 27 7 65 0 48 4 90 5 7 1 40 8 17\n1 66 4 25 0 62 2 84 9 13 6 64 7 46 8 59 5 19 3 85\n1 73 3 80 0 41 2 53 9 47 7 57 8 74 4 14 6 67 5 88\n5 64 3 84 6 46 1 78 0 84 7 26 8 28 9 52 2 41 4 63\n1 11 0 64 7 67 4 85 3 10 5 73 9 38 8 95 6 97 2 17\n4 60 8 32 2 95 3 93 1 65 6 85 7 43 9 85 5 46 0 59",
    "LA19": "10 10\n2 44 3 5 5 58 4 97 0 9 7 84 8 77 9 96 1 58 6 89\n4 15 7 31 1 87 8 57 0 77 3 85 2 81 5 39 9 73 6 21\n9 82 6 22 4 10 3 70 1 49 0 40 8 34 2 48 7 80 5 71\n1 91 2 17 7 62 5 75 8 47 4 11 3 7 6 72 9 35 0 55\n6 71 1 90 3 75 0 64 2 94 8 15 4 12 7 67 9 20 5 50\n7 70 5 93 8 77 2 29 4 58 6 93 3 68 1 57 9 7 0 52\n6 87 1 63 4 26 5 6 2 82 3 27 7 56 8 48 9 36 0 95\n0 36 5 15 8 41 9 78 3 76 6 84 4 30 7 76 2 36 1 8\n5 88 2 81 3 13 6 82 4 54 7 13 8 29 9 40 1 78 0 75\n9 88 4 54 6 64 7 32 0 52 2 6 8 54 5 82 3 6 1 26",
    "LA20": "10 10\n6 9 1 81 4 55 2 40 8 32 3 37 0 6 5 19 9 81 7 40\n7 21 2 70 9 65 4 64 1 46 5 65 8 25 0 77 3 55 6 15\n2 85 5 37 0 40 3 24 1 44 6 83 4 89 8 31 7 84 9 29\n4 80 6 77 7 56 0 8 2 30 5 59 3 38 1 80 9 41 8 97\n0 91 6 40 4 88 1 17 2 71 3 50 9 59 8 80 5 56 7 7\n2 8 6 9 3 58 5 77 1 29 8 96 0 45 9 10 4 54 7 36\n4 70 3 92 1 98 5 87 6 99 7 27 8 86 9 96 0 28 2 73\n1 95 7 92 3 85 4 52 6 81 9 32 8 39 0 59 2 41 5 56\n3 60 8 45 0 88 2 12 1 7 5 22 4 93 9 49 7 69 6 27\n0 21 2 61 3 68 5 26 6 82 9 71 8 44 4 99 7 33 1 84"
}

print(f'Loaded {len(INSTANCES)} instances')
print(f'Instances: {list(INSTANCES.keys())}')
print('Instance data loaded')

In [ ]:
def run_phase1():
    """Run Dynamic Programming on Phase 1 instances."""
    results = []
    
    # Check for existing results (resume support)
    completed = set()
    if os.path.exists(OUTPUT_PATH):
        df_existing = pd.read_csv(OUTPUT_PATH)
        completed = set(df_existing['instance'].values)
        results = df_existing.to_dict('records')
        logger.info(f"Resuming: {len(completed)} instances already completed")
    
    start_time = time.time()
    
    for idx, (name, data) in enumerate(INSTANCES.items(), 1):
        if name in completed:
            logger.info(f"[{idx}/{len(INSTANCES)}] {name}: SKIPPED (already completed)")
            continue
        
        logger.info(f"\n{'='*60}")
        logger.info(f"[{idx}/{len(INSTANCES)}] Solving {name} (BKS={BKS[name]})")
        logger.info(f"{'='*60}")
        
        try:
            # Parse instance
            lines = data.strip().split('\n')
            hdr = lines[0].strip().split()
            nj, nm = int(hdr[0]), int(hdr[1])
            
            # Build jobs list: jobs[j] = [(machine, processing_time), ...]
            jobs = []
            for j in range(nj):
                toks = lines[j+1].strip().split()
                job_ops = []
                for k in range(nm):
                    machine = int(toks[2*k])
                    ptime = int(toks[2*k+1])
                    job_ops.append((machine, ptime))
                jobs.append(job_ops)
            
            # Create instance (JSSPInstance class already defined in cell 4)
            instance = JSSPInstance(
                name=name,
                n_jobs=nj,
                n_machines=nm,
                jobs=jobs
            )
            
            # Solve with DP (DPSolver class already defined in cell 6)
            solver = DPSolver(instance, timeout=TIMEOUT)
            result = solver.solve(known_optimal=BKS[name])
            
            # Store result (DPResult uses best_makespan, not makespan)
            makespan = result.best_makespan if result.best_makespan is not None else -1
            gap = 0.0 if makespan == BKS[name] else ((makespan - BKS[name]) / BKS[name] * 100) if makespan > 0 else -1
            result_dict = {
                'instance': name,
                'size': f"{nj}x{nm}",
                'bks': BKS[name],
                'makespan': makespan,
                'gap_pct': gap,
                'optimal': result.optimal_proven,
                'time_s': result.computation_time_seconds,
                'states': result.states_explored
            }
            results.append(result_dict)
            
            # Save after each instance
            df = pd.DataFrame(results)
            df.to_csv(OUTPUT_PATH, index=False)
            
            # Log result
            status = "OPTIMAL" if result.optimal_proven else "TIMEOUT"
            logger.info(f"{status}: makespan={makespan}, gap={gap:.2f}%, time={result.computation_time_seconds:.1f}s")
            
        except Exception as e:
            logger.error(f"Error solving {name}: {e}")
            import traceback
            traceback.print_exc()
            results.append({
                'instance': name,
                'size': 'ERROR',
                'bks': BKS[name],
                'makespan': -1,
                'gap_pct': -1,
                'optimal': False,
                'time_s': -1,
                'states': -1
            })
    
    total_time = time.time() - start_time
    
    # Summary
    df = pd.DataFrame(results)
    optimal_count = df['optimal'].sum()
    
    logger.info(f"\n{'='*60}")
    logger.info(f"PHASE 1 COMPLETE")
    logger.info(f"{'='*60}")
    logger.info(f"Total instances: {len(INSTANCES)}")
    logger.info(f"Proven optimal: {optimal_count}")
    logger.info(f"Total time: {total_time/3600:.2f} hours")
    logger.info(f"Results saved to: {OUTPUT_PATH}")
    
    return df

# Run the experiment
print("Starting Phase 1 execution...")
results_df = run_phase1()
results_df